# GovernanceFund — 투표 → 비중 결정 알고리즘

**목적**: 투표 입력 → 포지션 비중(방향 + 레버리지) 산출 과정을 검증  
**구조**: Kahoot 스타일 UI 시뮬레이션 → 4가지 알고리즘 비교 → 공격적/보수적 프로파일

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.font_manager as fm
from matplotlib.patches import FancyBboxPatch
import warnings
warnings.filterwarnings('ignore')

# Windows 한국어 폰트 설정
_korean_fonts = ['Malgun Gothic', 'NanumGothic', 'AppleGothic', 'DejaVu Sans']
for _f in _korean_fonts:
    if any(_f.lower() in f.name.lower() for f in fm.fontManager.ttflist):
        plt.rcParams['font.family'] = _f
        print(f'✅ 폰트 설정: {_f}')
        break
plt.rcParams['axes.unicode_minus'] = False  # 마이너스 부호 깨짐 방지
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print('✅ 라이브러리 로드 완료')

✅ 폰트 설정: Malgun Gothic
✅ 라이브러리 로드 완료


## 1. 펀드 프로파일 설정

**공격적**: 빠른 반응, 큰 조정폭  
**보수적**: 느린 반응, 작은 조정폭

In [2]:
FUND_PROFILES = {
    'aggressive': {
        'MAX_STEP': 25,      # 한 투표 라운드에 최대 25%p 조정
        'ALPHA': 0.6,        # EMA 반응속도 (높을수록 빠름)
        'MIN_WEIGHT': 0,     # 코인별 최소 비중
        'MAX_WEIGHT': 80,    # 코인별 최대 비중
        'FUND_LEVERAGE': 5,  # 펀드 전체 레버리지
        'color': '#e74c3c'
    },
    'conservative': {
        'MAX_STEP': 10,
        'ALPHA': 0.25,
        'MIN_WEIGHT': 0,
        'MAX_WEIGHT': 60,
        'FUND_LEVERAGE': 2,
        'color': '#2ecc71'
    }
}

# 코인별 최근 30일 일간 변동성 (실측치 근사)
VOLATILITY = {
    'BTC':  0.030,
    'ETH':  0.040,
    'SOL':  0.055,
    'HYPE': 0.070,
    'BNB':  0.035,
}

COINS = list(VOLATILITY.keys())
print('📋 펀드 프로파일:')
for name, p in FUND_PROFILES.items():
    print(f"  {name:12s} | MAX_STEP={p['MAX_STEP']}%p | ALPHA={p['ALPHA']} | 레버리지={p['FUND_LEVERAGE']}x")

📋 펀드 프로파일:
  aggressive   | MAX_STEP=25%p | ALPHA=0.6 | 레버리지=5x
  conservative | MAX_STEP=10%p | ALPHA=0.25 | 레버리지=2x


## 2. 투표 시뮬레이션 (Kahoot 스타일)

각 참여자는 코인마다 방향 하나만 선택:  
`-2 강한 숏` / `-1 숏 늘려` / `0 유지` / `+1 롱 늘려` / `+2 강한 롱`

In [3]:
def simulate_votes(participants, coins):
    """
    participants: [{'name': str, 'deposit': float, 'votes': {coin: int(-2~+2)}}, ...]
    반환: {coin: {'score': float, 'breakdown': dict}}
    """
    total_deposit = sum(p['deposit'] for p in participants)
    results = {}

    for coin in coins:
        weighted_score = 0
        breakdown = {-2: 0, -1: 0, 0: 0, 1: 0, 2: 0}
        for p in participants:
            vote = p['votes'].get(coin, 0)
            weight = p['deposit'] / total_deposit
            weighted_score += vote * weight
            breakdown[vote] += p['deposit']
        results[coin] = {
            'score': weighted_score / 2,  # -1 ~ +1 정규화
            'breakdown': breakdown,
            'total': total_deposit
        }
    return results


def print_vote_result(vote_result):
    print(f"{'코인':<6} {'score':>8}  방향  {'분포 (예치금 기준)':<40}")
    print('-' * 70)
    for coin, r in vote_result.items():
        score = r['score']
        direction = '🟢 롱▲' if score > 0.1 else ('🔴 숏▼' if score < -0.1 else '⚪ 유지')
        bar = ''
        labels = {-2:'강숏', -1:'숏', 0:'유지', 1:'롱', 2:'강롱'}
        for k in [-2, -1, 0, 1, 2]:
            pct = r['breakdown'][k] / r['total'] * 100
            if pct > 0:
                bar += f"{labels[k]}:{pct:.0f}% "
        print(f"{coin:<6} {score:>+8.3f}  {direction}  {bar}")


# ── 예시 투표 시나리오 ──────────────────────────────────────────
example_participants = [
    {'name': 'Alice', 'deposit': 500,
     'votes': {'BTC': 2, 'ETH': 1, 'SOL': -1, 'HYPE': 2, 'BNB': 0}},
    {'name': 'Bob',   'deposit': 300,
     'votes': {'BTC': 1, 'ETH': 0, 'SOL': 1,  'HYPE': 1, 'BNB': -1}},
    {'name': 'Carol', 'deposit': 150,
     'votes': {'BTC': 0, 'ETH': -1, 'SOL': 2, 'HYPE': -1, 'BNB': 1}},
    {'name': 'Dave',  'deposit': 50,
     'votes': {'BTC': -1, 'ETH': 2, 'SOL': 0, 'HYPE': 0, 'BNB': 2}},
]

vote_result = simulate_votes(example_participants, COINS)
print('📊 투표 결과:'); print_vote_result(vote_result)

📊 투표 결과:
코인        score  방향  분포 (예치금 기준)                             
----------------------------------------------------------------------
BTC      +0.625  🟢 롱▲  숏:5% 유지:15% 롱:30% 강롱:50% 
ETH      +0.225  🟢 롱▲  숏:15% 유지:30% 롱:50% 강롱:5% 
SOL      +0.050  ⚪ 유지  숏:50% 유지:5% 롱:30% 강롱:15% 
HYPE     +0.575  🟢 롱▲  숏:15% 유지:5% 롱:30% 강롱:50% 
BNB      -0.025  ⚪ 유지  숏:30% 유지:50% 롱:15% 강롱:5% 


## 3. 비중 업데이트 알고리즘 4가지 비교

In [4]:
def algo_fixed_step(current, score, coin, profile):
    """방법 1: 고정 STEP — 단순 선형 (증분)"""
    step = profile['MAX_STEP']
    return current + score * step


def algo_conviction(current, score, coin, profile):
    """방법 2: 확신도 비례 — 만장일치일수록 크게 움직임 (증분)"""
    conviction = abs(score)  # 0~1
    step = profile['MAX_STEP'] * conviction
    return current + score * step


def algo_vol_adjusted(current, score, coin, profile):
    """방법 3: 변동성 연동 — 위험한 코인은 조심스럽게 (증분)"""
    vol = VOLATILITY.get(coin, 0.05)
    vol_factor = 0.03 / vol  # BTC 변동성 기준으로 정규화
    conviction = abs(score)
    step = profile['MAX_STEP'] * conviction * vol_factor
    return current + score * step


def algo_combined(current, score, coin, profile):
    """방법 4: 통합 증분 (확신도 + 변동성 + 독식방지 + EMA)"""
    vol = VOLATILITY.get(coin, 0.05)
    vol_factor = 0.03 / vol
    conviction = abs(score)
    concentration_limit = 1 - (current / 100)  # 비중 클수록 제한
    concentration_limit = max(0.1, concentration_limit)

    raw_target = current + score * profile['MAX_STEP'] * conviction * vol_factor * concentration_limit

    # EMA 완충
    alpha = profile['ALPHA']
    return alpha * raw_target + (1 - alpha) * current


def algo_target(current, score, coin, profile):
    """
    ★ 방법 5: 목표(TARGET) 방식 — 최종 채택 로직 ★

    증분 방식(1~4)의 치명적 한계: 방향 전환이 느려서, 미래를 알아도
    롱→숏 전환에 3~4주 걸림 → 시점을 놓침.

    target 방식: score가 곧바로 '목표 포지션'을 결정.
      score = +1 → 풀 롱 목표
      score = -1 → 풀 숏 목표 (즉시 음수 영역 진입 가능)
    EMA(alpha)는 목표로 가는 '속도'만 조절.

    ※ 실제 운용/컨트랙트에 들어가는 로직은 이것.
       단, 실제로는 모든 코인을 한꺼번에 정규화(sum(abs)=100)하므로
       아래는 단일 코인 응답곡선 시각화용 근사.
    """
    vol = VOLATILITY.get(coin, 0.05)
    vol_factor = 0.03 / vol
    # score → 목표 비중 (MAX_WEIGHT까지 스케일, 부호 유지)
    raw_target = score * vol_factor * profile['MAX_WEIGHT']
    raw_target = np.clip(raw_target, -profile['MAX_WEIGHT'], profile['MAX_WEIGHT'])

    alpha = profile['ALPHA']
    return alpha * raw_target + (1 - alpha) * current


ALGORITHMS = {
    'Fixed Step':   algo_fixed_step,
    'Conviction':   algo_conviction,
    'Vol-Adjusted': algo_vol_adjusted,
    'Combined':     algo_combined,
    'Target ★':     algo_target,   # 최종 채택
}


def apply_algo(current_weights, vote_result, profile, algo_fn):
    """알고리즘 적용 후 합계 100% 정규화, 범위 클리핑"""
    raw = {}
    for coin in COINS:
        score = vote_result[coin]['score']
        new_w = algo_fn(current_weights[coin], score, coin, profile)
        raw[coin] = np.clip(new_w, profile['MIN_WEIGHT'], profile['MAX_WEIGHT'])

    # 합계 100% 정규화
    total = sum(abs(v) for v in raw.values())
    if total == 0:
        return {c: 0 for c in COINS}
    return {c: raw[c] / total * 100 for c in COINS}


# 현재 포트폴리오 (초기 동일 비중)
current_weights = {coin: 20.0 for coin in COINS}

print(f"{'':20s}" + "".join(f"{c:>8}" for c in COINS) + "  (합계)")
print(f"{'현재 비중':20s}" + "".join(f"{current_weights[c]:>7.1f}%" for c in COINS))
print('-' * 70)
for profile_name, profile in FUND_PROFILES.items():
    for algo_name, algo_fn in ALGORITHMS.items():
        new_w = apply_algo(current_weights, vote_result, profile, algo_fn)
        row = f"[{profile_name[:4]}] {algo_name:15s}"
        vals = "".join(f"{new_w[c]:>7.1f}%" for c in COINS)
        total = sum(new_w.values())
        print(f"{row:20s}{vals}  ({total:.0f}%)")

                         BTC     ETH     SOL    HYPE     BNB  (합계)
현재 비중                  20.0%   20.0%   20.0%   20.0%   20.0%
----------------------------------------------------------------------
[aggr] Fixed Step        26.1%   18.8%   15.6%   25.2%   14.2%  (100%)
[aggr] Conviction        24.9%   17.8%   16.8%   23.7%   16.7%  (100%)
[aggr] Vol-Adjusted      26.0%   18.3%   17.5%   20.6%   17.5%  (100%)
[aggr] Combined          23.1%   19.1%   18.7%   20.3%   18.7%  (100%)
[aggr] Target ★          42.1%   17.8%   10.3%   22.0%    7.7%  (100%)
[cons] Fixed Step        22.9%   19.4%   17.9%   22.5%   17.2%  (100%)
[cons] Conviction        22.2%   19.0%   18.6%   21.6%   18.6%  (100%)
[cons] Vol-Adjusted      22.6%   19.3%   18.9%   20.3%   18.9%  (100%)
[cons] Combined          20.5%   19.8%   19.8%   20.1%   19.8%  (100%)
[cons] Target ★          26.9%   19.3%   17.0%   20.6%   16.2%  (100%)


## 4. 롱/숏 + 레버리지 포지션 산출

In [5]:
def weights_to_positions(weights, vote_result, tvl, profile):
    """
    비중 + 투표 방향 → 실제 노셔널 포지션 산출
    score > 0 : 롱, score < 0 : 숏, score ≈ 0 : 스킵
    """
    leverage = profile['FUND_LEVERAGE']
    positions = {}

    for coin in COINS:
        score = vote_result[coin]['score']
        weight_pct = weights[coin]

        if weight_pct < 1.0:  # 1% 미만 스킵 (수수료 낭비)
            positions[coin] = {'side': 'skip', 'notional': 0, 'weight': 0}
            continue

        side = 'long' if score >= 0 else 'short'
        notional = (weight_pct / 100) * tvl * leverage

        positions[coin] = {
            'side': side,
            'weight': weight_pct,
            'notional': notional,
            'score': score,
        }
    return positions


TVL = 100_000  # USDC

# Combined 알고리즘 + 두 프로파일로 포지션 산출
for profile_name, profile in FUND_PROFILES.items():
    new_w = apply_algo(current_weights, vote_result, profile, algo_combined)
    positions = weights_to_positions(new_w, vote_result, TVL, profile)

    print(f"\n{'='*60}")
    print(f"프로파일: {profile_name.upper()} | TVL: ${TVL:,} | 레버리지: {profile['FUND_LEVERAGE']}x")
    print(f"{'코인':<6} {'방향':<6} {'비중':>7} {'노셔널':>12} {'score':>8}")
    print('-' * 50)
    total_notional = 0
    for coin, pos in positions.items():
        if pos['side'] == 'skip':
            continue
        side_emoji = '🟢롱' if pos['side'] == 'long' else '🔴숏'
        print(f"{coin:<6} {side_emoji:<6} {pos['weight']:>6.1f}%  "
              f"${pos['notional']:>10,.0f}  {pos['score']:>+8.3f}")
        total_notional += pos['notional']
    print(f"{'합계':>14} {100:>6.0f}%  ${total_notional:>10,.0f}")


프로파일: AGGRESSIVE | TVL: $100,000 | 레버리지: 5x
코인     방향          비중          노셔널    score
--------------------------------------------------
BTC    🟢롱       23.1%  $   115,520    +0.625
ETH    🟢롱       19.1%  $    95,718    +0.225
SOL    🟢롱       18.7%  $    93,663    +0.050
HYPE   🟢롱       20.3%  $   101,543    +0.575
BNB    🔴숏       18.7%  $    93,556    -0.025
            합계    100%  $   500,000

프로파일: CONSERVATIVE | TVL: $100,000 | 레버리지: 2x
코인     방향          비중          노셔널    score
--------------------------------------------------
BTC    🟢롱       20.5%  $    41,093    +0.625
ETH    🟢롱       19.8%  $    39,698    +0.225
SOL    🟢롱       19.8%  $    39,554    +0.050
HYPE   🟢롱       20.1%  $    40,109    +0.575
BNB    🔴숏       19.8%  $    39,546    -0.025
            합계    100%  $   200,000


## 5. 알고리즘별 반응 비교 시각화

In [6]:
scores = np.linspace(-1, 1, 200)
current = 30.0

algo_colors = {
    'Fixed Step':   '#3498db',
    'Conviction':   '#e74c3c',
    'Vol-Adjusted': '#2ecc71',
    'Combined':     '#f39c12',
    'Target ★':     '#9b59b6',   # 최종 채택 — 보라색 강조
}

fig, axes = plt.subplots(2, len(COINS), figsize=(20, 10))
fig.suptitle('알고리즘별 Score → 비중 변화 반응 곡선 (코인별, 현재 비중 30%)\n'
             '★ Target(보라)만 음수 영역=숏으로 진입 가능 — 나머지 증분 방식은 전환 느림',
             fontsize=13, fontweight='bold')

for p_idx, (profile_name, profile) in enumerate(FUND_PROFILES.items()):
    for c_idx, coin in enumerate(COINS):
        ax = axes[p_idx][c_idx]

        for algo_name, algo_fn in ALGORITHMS.items():
            y = [algo_fn(current, s, coin, profile) for s in scores]
            lw = 2.5 if algo_name == 'Target ★' else 1.8
            ax.plot(scores, y, label=algo_name,
                    color=algo_colors[algo_name], linewidth=lw)

        ax.axhline(y=current, color='gray', linestyle='--', alpha=0.5, linewidth=1)
        ax.axhline(y=0,       color='black', linestyle='-',  alpha=0.4, linewidth=0.8)
        ax.axvline(x=0,       color='gray', linestyle=':',  alpha=0.4)
        ax.fill_betweenx([-80, 100], -1, 0, alpha=0.03, color='red')
        ax.fill_betweenx([-80, 100], 0,  1, alpha=0.03, color='green')

        vol_pct = VOLATILITY[coin] * 100
        ax.set_title(f'{coin}  (변동성 {vol_pct:.1f}%)\n{profile_name.upper()}', fontsize=10)
        ax.set_xlim(-1, 1)
        ax.set_ylim(-80, 100)
        ax.set_xlabel('투표 Score', fontsize=9)
        if c_idx == 0:
            ax.set_ylabel(f'{profile_name}\n비중 (%)  (음수=숏)', fontsize=9)

        if p_idx == 0 and c_idx == len(COINS) - 1:
            ax.legend(fontsize=8, loc='upper left')

plt.tight_layout()
plt.savefig('algo_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ algo_comparison.png 저장')
print()
print('📌 핵심 관찰:')
print('   • Target(보라)만 score<0 에서 비중이 0 아래(숏)로 내려감')
print('   • 증분 방식 1~4는 current(30%) 근처에서만 찔끔 움직임 → 숏 전환 불가')
print('   • BTC에서 Conviction=Vol-Adjusted 겹침은 정상 (vol_factor=1)')

✅ algo_comparison.png 저장

📌 핵심 관찰:
   • Target(보라)만 score<0 에서 비중이 0 아래(숏)로 내려감
   • 증분 방식 1~4는 current(30%) 근처에서만 찔끔 움직임 → 숏 전환 불가
   • BTC에서 Conviction=Vol-Adjusted 겹침은 정상 (vol_factor=1)


## 6. 다회차 투표 시뮬레이션 — 비중 변화 추적

In [7]:
np.random.seed(42)

def random_vote_scenario(participants, coins, scenario='random'):
    """시나리오별 투표 생성"""
    for p in participants:
        for coin in coins:
            if scenario == 'random':
                p['votes'][coin] = np.random.choice([-2, -1, 0, 1, 2],
                                                     p=[ 0.1, 0.2, 0.4, 0.2, 0.1])
            elif scenario == 'bullish':
                p['votes'][coin] = np.random.choice([0, 1, 2], p=[0.2, 0.4, 0.4])
            elif scenario == 'bearish':
                p['votes'][coin] = np.random.choice([-2, -1, 0], p=[0.4, 0.4, 0.2])
    return participants


ROUNDS = 12  # 12주 (분기)
scenarios_to_test = ['random', 'bullish', 'bearish']
scenario_labels = {'random': '랜덤 투표', 'bullish': '강세장 투표', 'bearish': '약세장 투표'}

fig, axes = plt.subplots(len(FUND_PROFILES), len(scenarios_to_test),
                          figsize=(18, 10))
fig.suptitle('다회차 투표 — 포트폴리오 비중 변화 추적', fontsize=15, fontweight='bold')

coin_colors = {'BTC': '#F7931A', 'ETH': '#627EEA', 'SOL': '#9945FF',
               'HYPE': '#00D4FF', 'BNB': '#F3BA2F'}

for p_idx, (profile_name, profile) in enumerate(FUND_PROFILES.items()):
    for s_idx, scenario in enumerate(scenarios_to_test):
        ax = axes[p_idx][s_idx]

        weights_history = {coin: [20.0] for coin in COINS}
        current_w = {coin: 20.0 for coin in COINS}

        participants = [
            {'name': 'A', 'deposit': 500, 'votes': {}},
            {'name': 'B', 'deposit': 300, 'votes': {}},
            {'name': 'C', 'deposit': 200, 'votes': {}},
        ]

        for round_i in range(ROUNDS):
            participants = random_vote_scenario(participants, COINS, scenario)
            vr = simulate_votes(participants, COINS)
            new_w = apply_algo(current_w, vr, profile, algo_combined)
            current_w = new_w
            for coin in COINS:
                weights_history[coin].append(new_w[coin])

        for coin in COINS:
            ax.plot(weights_history[coin], label=coin,
                    color=coin_colors[coin], linewidth=2, marker='o', markersize=3)

        ax.axhline(y=20, color='gray', linestyle='--', alpha=0.4, linewidth=1)
        ax.set_ylim(0, 80)
        ax.set_xlabel('투표 라운드 (주)')
        ax.set_ylabel('비중 (%)')
        ax.set_title(f'{profile_name.upper()} | {scenario_labels[scenario]}')
        if p_idx == 0 and s_idx == len(scenarios_to_test) - 1:
            ax.legend(loc='upper right', fontsize=8)

plt.tight_layout()
plt.savefig('multiround_weights.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ multiround_weights.png 저장')

✅ multiround_weights.png 저장


## 7. 알고리즘 선택 가이드

| 알고리즘 | 방식 | 특징 | 한계 |
|----------|------|------|------|
| Fixed Step | 증분 | 단순, 예측 가능 | 방향 전환 느림 |
| Conviction | 증분 | 만장일치 강조 | 방향 전환 느림 |
| Vol-Adjusted | 증분 | 리스크 균형 | 방향 전환 느림 |
| Combined | 증분 | 4요소 통합 | 곱하기 과다 → 신호 약화 |
| **★ Target** | **목표** | **score→목표 직결, 숏 즉시 진입** | **최종 채택** |

### 왜 Target 방식인가 (02번 백테스트로 검증됨)

증분(1~4) 방식은 매 라운드 `current`에서 조금씩만 이동 →  
미래를 완벽히 알아도(Perfect 시나리오) 롱→숏 전환에 3~4주 걸려 **타이밍 실종**.

→ 02번 백테스트에서 증분 방식은 Perfect조차 -177% 손실.  
→ Target 방식으로 바꾼 뒤 Perfect가 압도적 우위(검증 통과).

**∴ 실제 운용/컨트랙트에 들어가는 로직은 Target 방식.**

```
target[coin] = score × vol_factor      # 방향+크기 즉시 결정
정규화: sum(abs) = 100
new = EMA(target, current; alpha)       # alpha=수렴 속도
```

### 파라미터 설명 (플랫폼 문서용)

- **ALPHA**: 목표 수렴 속도. 0에 가까울수록 천천히(보수적), 1에 가까울수록 즉각(공격적).
- **MAX_WEIGHT**: 코인 1종당 최대 비중 상한 (롱/숏 공통, 독식 방지).
- **FUND_LEVERAGE**: 전체 포트폴리오 레버리지 배수.
- ~~MAX_STEP~~: 증분 방식 전용. Target 방식에선 미사용 (ALPHA가 대체).

In [8]:
# ── 최종 출력: 투표 → 포지션 전체 플로우 요약 ──────────────────
print('=' * 65)
print('  GovernanceFund 투표 → 포지션 플로우 (Combined 알고리즘)')
print('=' * 65)

profile = FUND_PROFILES['aggressive']
new_w = apply_algo(current_weights, vote_result, profile, algo_combined)
positions = weights_to_positions(new_w, vote_result, TVL, profile)

print(f"\nTVL: ${TVL:,} USDC | 레버리지: {profile['FUND_LEVERAGE']}x")
print(f"총 노셔널: ${TVL * profile['FUND_LEVERAGE']:,} USDC\n")

print(f"{'코인':<6} {'투표score':>10} {'비중':>8} {'방향':>6} {'노셔널':>14}")
print('-' * 55)
for coin in COINS:
    pos = positions[coin]
    score = vote_result[coin]['score']
    side = '🟢 LONG' if pos['side'] == 'long' else ('🔴 SHORT' if pos['side'] == 'short' else '  SKIP')
    print(f"{coin:<6} {score:>+10.3f} {pos['weight']:>7.1f}%  {side}  ${pos['notional']:>12,.0f}")

  GovernanceFund 투표 → 포지션 플로우 (Combined 알고리즘)

TVL: $100,000 USDC | 레버리지: 5x
총 노셔널: $500,000 USDC

코인        투표score       비중     방향            노셔널
-------------------------------------------------------
BTC        +0.625    23.1%  🟢 LONG  $     115,520
ETH        +0.225    19.1%  🟢 LONG  $      95,718
SOL        +0.050    18.7%  🟢 LONG  $      93,663
HYPE       +0.575    20.3%  🟢 LONG  $     101,543
BNB        -0.025    18.7%  🔴 SHORT  $      93,556
